# train — Direction 1: SHAC through the replica, on two GPUs, packaged for submission

One GPU T4 ×2 session, GPU work only (`hvac_controller/d1/train.py`, `policy.py`); the winner's score in the
organisers' simulator comes afterwards from `evaluate.ipynb`.

1. **smoke** (≈ 15 min, GPU 0), stops the notebook if anything fails:
   - parity of the submission path (`KagglePolicy` on the organisers' simulator, CPU, 36 steps) with the training
     path (GPU, in a padded batch with another building);
   - the padded multi-building environment against each building alone;
   - checkpointed against plain gradients;
   - a short run of the full configuration: seconds per update and memory.
2. **training**, both GPUs at once, `MINUTES` each (cut so that the session's 12 h hold the runs' setup, burn-in,
   final evaluations and the packaging). The two runs share data and settings and differ in one thing:
   - `main` (GPU 0): with the weather and zone-temperature memory;
   - `no_memory` (GPU 1): without it (the ablation).
   - Training: the shipped scenarios outside the held-out building (building_train_2) and the generated
     `gen_train` ones. Held out: building_train_2's shipped strips, building_train_2 × 2 new stations and the
     `gen_val` ones (6 held-out generated buildings × 2 strips).
   - Each run keeps its best checkpoint on the held-out replica score (30 days, the averaged weights).
3. **winner** = the run with the lower held-out replica score.
4. **package:** `/kaggle/working/hvac-submission.zip` (flat: `d1_policy.pt`, `d1_policy.py`, `README.txt`), the next
   version of the `hvac-submission` dataset (`evaluate.ipynb` scores it, `submit.ipynb` submits it); and
   `/kaggle/working/train_logs_<VERSION>.zip`.

**Inputs:** the competition data and `buildings`. Accelerator **GPU T4 ×2**, Internet on. ≈ 11.5 h.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
VERSION = "v2"            # names the package's README and the logs' zip
SETTINGS = {"h": 288, "gamma": 0.998, "seg": 24, "buckets": 3, "winsor": 10.0, "spike": 10.0, "ema": 0.98,
            "eval_minutes": 20, "eval_seeds": 1}
RUNS = {"main": {"memory": 1}, "no_memory": {"memory": 0}}
MINUTES = 630             # training budget of each run, cut to fit the session before the runs start
SESSION_HOURS = 12        # Kaggle's limit for a GPU session

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch

T0 = time.time()
assert torch.cuda.device_count() >= 2, "set the accelerator to GPU T4 x2"
print(torch.__version__, torch.cuda.device_count(), "x", torch.cuda.get_device_name())
REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/train")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")


def dataset(slug):
    """One of the four datasets (docs/KAGGLE.md), wherever Kaggle mounts it."""
    return next(p for p in (INPUT / "datasets" / "cryandrrich" / slug, INPUT / slug) if p.is_dir())


SHIPPED = next(p for p in sorted(INPUT.rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)  # the smoke's parity
DATA = dataset("buildings")
MANIFEST = json.loads((DATA / "manifest_v2.json").read_text())
CACHES = {p.stem: p for p in sorted(DATA.glob("*.pkl"))}
held_out = {e["building"] for e in MANIFEST if e["tag"] == "shipped_building_new_station"}  # building_train_2
shipped = sorted(f"{p.parent.parent.name}__{p.parent.name}" for p in SHIPPED.glob("*/*/config.gin"))
TRAIN = ([s for s in shipped if s.split("__")[0] not in held_out]
         + [e["label"] for e in MANIFEST if e["tag"] == "gen_train"])
VAL = ([s for s in shipped if s.split("__")[0] in held_out]
       + [e["label"] for e in MANIFEST if e["tag"] in ("gen_val", "shipped_building_new_station")])
missing = [s for s in TRAIN + VAL if s not in CACHES]
assert not missing, f"replica caches missing from buildings: {missing[:10]}"
print(f"{len(TRAIN)} training, {len(VAL)} held-out scenarios; {len(CACHES)} caches")
WORK.mkdir(parents=True, exist_ok=True)  # outside /kaggle/working: the output is the two zips alone


def train_cmd(out_dir, settings, extra=()):
    cmd = [sys.executable, "-m", "hvac_controller.d1.train", str(out_dir),
           "--caches", ",".join(str(CACHES[s]) for s in TRAIN), "--val-caches", ",".join(str(CACHES[s]) for s in VAL)]
    for k, v in settings.items():
        cmd += [f"--{k.replace('_', '-')}", str(v)]
    return cmd + list(extra)


def tail(path, n=3000):
    text = Path(path).read_text() if Path(path).exists() else ""
    return text[-8000:] if "Traceback" in text else text[-n:]

In [ ]:
# 1. Smoke: start-up checks and a short run of the full configuration on GPU 0. Stops here if anything is wrong.
smoke = WORK / "smoke"
smoke.mkdir(exist_ok=True)
t0 = time.time()
with open(smoke / "log.txt", "w") as log:
    rc = subprocess.run(train_cmd(smoke, {**SETTINGS, "minutes": 4, "eval_minutes": 1000, "eval_steps": 288}, ["--smoke"]),
                        cwd=REPO, env=dict(ENV, CUDA_VISIBLE_DEVICES="0"), stdout=log, stderr=subprocess.STDOUT).returncode
print(f"smoke exit {rc} in {time.time() - t0:.0f}s\n" + tail(smoke / "log.txt", 6000))
rep = json.loads((smoke / "report.json").read_text()) if (smoke / "report.json").exists() else {}
assert rc == 0 and all(rep.get(k, {}).get("ok") for k in ("parity", "group", "checkpoint")), "smoke failed: read the log"
rows = [json.loads(l) for l in (smoke / "train_log.jsonl").read_text().splitlines()]
s_it = sum(r["s"] for r in rows[1:]) / max(len(rows) - 1, 1) if len(rows) > 1 else rows[0]["s"]
print(f"{len(rows)} updates, {s_it:.0f} s per update, peak {max(r['mem_gb'] for r in rows):.1f} GB: "
      f"about {MINUTES * 60 / s_it:.0f} updates in {MINUTES} min (want >= 150)")

In [ ]:
# 2. The two runs, one per GPU; evaluations, skipped spikes and rollbacks are echoed as they happen. Training gets
# what is left of the session after 45 min for the runs' setup, burn-in and final evaluations and the packaging.
minutes = min(MINUTES, int((SESSION_HOURS * 3600 - (time.time() - T0) - 45 * 60) / 60))
print(f"training {minutes} min per run ({time.time() - T0:.0f}s of the session used)")
assert minutes >= 240, "not enough session left for a useful run"
t0, procs = time.time(), {}
for gpu, (name, extra) in enumerate(RUNS.items()):
    d = WORK / name
    d.mkdir(exist_ok=True)
    log = open(d / "log.txt", "w")
    procs[name] = (subprocess.Popen(train_cmd(d, {**SETTINGS, **extra, "minutes": minutes}), cwd=REPO, stdout=log,
                                    stderr=subprocess.STDOUT, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))), log, d)
    print(f"GPU {gpu}: {name} {extra}", flush=True)
seen = {name: 0 for name in procs}
while True:
    running = any(p.poll() is None for p, _, _ in procs.values())
    for name, (p, _, d) in procs.items():
        lines = (d / "log.txt").read_text().splitlines()
        for line in lines[seen[name]:]:
            if line.startswith(("[eval]", "Traceback")) or any(w in line for w in ("restored", "skipped", "bucket(s)",
                                                                                     "out of memory")):
                print(f"[{(time.time() - t0) / 60:5.0f} min] {name}: {line[:400]}", flush=True)
        seen[name] = len(lines)
    if not running:
        break
    time.sleep(60)
REPORTS = {}
for name, (p, log, d) in procs.items():
    log.close()
    REPORTS[name] = json.loads((d / "report.json").read_text()) if (d / "report.json").exists() else {}
    print(f"===== {name}: exit {p.returncode}, best held-out {REPORTS[name].get('best')}, "
          f"{REPORTS[name].get('iterations')} updates, {REPORTS[name].get('spikes')} skipped spikes, "
          f"{REPORTS[name].get('rollbacks')} rollbacks\n" + tail(d / "log.txt", 2500))
done = {n: r["best"] for n, r in REPORTS.items() if r.get("best") is not None and (WORK / n / "d1_policy.pt").exists()}
WINNER = min(done, key=done.get)
print("winner:", WINNER, done)

In [ ]:
# 3. The package (the next hvac-submission version) and the logs.
readme = (f"Direction 1 {VERSION} policy ({WINNER}), commit {COMMIT}: trained on {len(TRAIN)} scenarios, chosen on "
          f"{len(VAL)} held-out ones (replica {done[WINNER]:.5f}; all runs {json.dumps(done)}). d1_policy.py = "
          f"hvac_controller/d1/policy.py. evaluate.ipynb scores it in the organisers' simulator, submit.ipynb submits it.\n")
name = Path("/kaggle/working/hvac-submission.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(WORK / WINNER / "d1_policy.pt", "d1_policy.pt")
    z.write(REPO / "hvac_controller" / "d1" / "policy.py", "d1_policy.py")
    z.writestr("README.txt", readme)
print(readme, "wrote", name, zipfile.ZipFile(name).namelist())
name = Path(f"/kaggle/working/train_logs_{VERSION}.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in WORK.rglob("*"):
        if p.is_file() and (p.suffix != ".pt" or p.name == "d1_policy.pt"):
            z.write(p, p.relative_to(WORK))
print("wrote", name)